In [1]:
import os
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, OpenAIChatCompletionsModel, trace, function_tool

In [3]:
load_dotenv(override=True)

True

In [4]:
# Create Groq client (OpenAI-compatible)
client = AsyncOpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

In [5]:
# Attach model
model = OpenAIChatCompletionsModel(
    model="openai/gpt-oss-20b",   # Groq model name
    openai_client=client,
)

In [6]:
agent = Agent(
    name="GroqAgent",
    instructions="You are a helpful AI assistant",
    model=model,
)

In [7]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [8]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=model
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=model
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=model
)

In [9]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify SOC 2 Compliance and Audit Prep with AI‑Powered Automation

Hi [First Name],

I know that maintaining SOC 2 compliance and preparing for audits is a high‑stakes, time‑consuming process for most security teams. Manual gap analyses, continuous monitoring, and documentation all add up to missed deadlines and costly re‑work.

ComplAI turns that process on its head. Our AI‑powered SaaS platform:

| Feature | What You Gain |
|---------|---------------|
| **Automated Gap Identification** | Detects compliance gaps in real time, no manual scans |
| **Continuous Monitoring & Alerting** | Keeps you audit‑ready 24/7 with instant remediation insights |
| **Audit‑Ready Reporting** | Generates full SOC 2 evidence bundles in minutes |
| **Resource Savings** | Cut audit preparation time by 35‑50 % and reduce manual effort |

All of this is delivered through a single, user‑friendly dashboard that your team can start using in less than 24 hours.

Would you be open to a 15‑minute demo ne

In [10]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

**Subject:** Streamline SOC 2 Compliance—AI‑Powered Audits in 30 Days

Hi [First Name],

I know keeping SOC 2 under control can feel like a moving target—continuous monitoring, evidence collection, and audit readiness all at once.  
At **ComplAI** we turned that juggling act into a single, AI‑driven platform that delivers:

| What you need | How ComplAI delivers |
|---------------|---------------------|
| **Real‑time compliance status** | AI scans logs & config changes 24/7, flagging gaps instantly |
| **Audit‑ready evidence** | Auto‑generated artifacts that auditors love—no manual uploads |
| **Time‑to‑audit 70 % faster** | Our platform maps controls to audit questions automatically |
| **Cost savings** | Reduce audit fees and internal audit hours by up to 50 % |

We’ve helped companies in fintech and SaaS cut audit cycle time from months to weeks while keeping risk below 1 %—all without adding a full‑time compliance team.

Could we schedule a 20‑minute demo next week to show how Comp

In [11]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=model
)

In [12]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

OPENAI_API_KEY is not set, skipping trace export


Best sales email:
Subject: Cut Your SOC 2 Audit Time in Half with AI‑Driven Compliance

Hi [First Name],

I know that SOC 2 audits can feel like a full‑time job: gathering evidence, mapping controls, and chasing missing documentation. It’s costly, time‑consuming, and often leaves teams scrambling at the last minute.

ComplAI changes that. Our AI‑powered platform:

* **Automates evidence collection** – Pulls logs, policies, and configuration data from your stack in real time.  
* **Maps controls to SOC 2 requirements** – Zero manual mapping, only a few clicks to see gaps.  
* **Provides an audit‑ready audit trail** – Every change is logged, versioned, and ready for review.  
* **Reduces audit cycle time by up to 70 %** – Teams finish in weeks, not months.

We’ve helped companies like [Client] complete their first SOC 2 audit in 6 weeks, saving $35k on consultant fees.

Could we schedule a 15‑minute call next week to show how ComplAI fits into your current tooling stack? Please let me kn

### Create tool to send email

In [13]:
@function_tool
def send_email(body: str):
    """Send email using this tool"""
    return {"status": "sucess"}

In [14]:
send_email

FunctionTool(name='send_email', description='Send email using this tool', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001E39E669FD0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None)

In [15]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools


[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001E39E73C950>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunc

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=model)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)